<img src=https://aggis.org/images/aggis.ico> 

# North American Ski Explorer
## `ski.ipynb`
### aggis.org/files/notebooks/.
Adapted from assignment submission for UCLA GEOG 412: Programming for Geospatial Data Science, Unit 5.
<hr/>

#### 📦 Packages

In [ ]:
!pip install geopy pandas geopandas geoplot mapclassify contextily rasterio keplergl
!sudo apt install -y libspatialindex-dev
!pip install rtree

import math
import numpy as np
import requests
import pandas as pd
import geopandas as gpd
import rtree
import rasterio
import rasterio.mask
import mapclassify
import matplotlib.pyplot as plt
import matplotlib.colors as cl
import contextily as cx
import ipywidgets as widgets
import folium
import xyzservices.providers as xyz

from keplergl import KeplerGl
import copy
from google.colab import output
output.enable_custom_widget_manager()

from math import radians, cos, sin, asin, sqrt, degrees, pi, atan2
from geopy.geocoders import Nominatim
from rasterio.plot import show
from IPython.display import display, clear_output
from shapely.geometry import shape, Point

#### 📊 Data

In [ ]:
# For Google Colab =====================================================|
!wget = https://aggis.org/files/notebooks/data/clean-na_skiresorts.csv #|
!wget = https://aggis.org/files/notebooks/data/temperature.tif         #|
!wget = https://aggis.org/files/notebooks/data/masked_na_dtm.tif       #|
na_skiresorts_path = "clean-na_skiresorts.csv"                         #|
temperature_path = "temperature.tif"                                   #|
elevation_path = "masked_na_dtm.tif"                                   #|
# ======================================================================|
# For Jupyter Notebook =================================================|
# (local clone of the repo)                                             |
#na_skiresorts_path = "data/clean-na_skiresorts.csv"                    |
#temperature_path = "data/temperature.tif"                              |
#elevation_path = "data/masked_na_dtm.tif"                              |
# ======================================================================|

na_skiresorts = pd.read_csv(na_skiresorts_path)
temperature = rasterio.open(temperature_path)
elevation = rasterio.open(elevation_path)

#### 🔨 Tool: North American Ski Explorer

**Thanks for trying version 0.1 of the North American Ski Explorer!**
**Please read directions below:**

North American Ski Explorer v0.1 allows users to search for ski resorts and recreation areas within a specified radius of a specified point or place.

This tool receives a place from the user which will be geocoded. Both street addresses and capitalized place names are valid inputs for the geocoder. The tool then receives the user's buffer radius, which is entered in meters. Valid buffer radiuses range from 8,000m to 130,000m (8 km or 130km).

After confirming the validity of your inputs for "Address" and "Radius", click "Go!" to execute your search. If your specified search zone does not return any results, you will be asked to increase your radius or change your location.

If your search returns results, the tool will ask you to select a visualization via dropdown menu. The initially selected value should always be "Interactive map (suggested)", but this visualization will not display; to display a visualization, change the value in the dropdown menu to generate that visualization. If the "Interactive map" is what you're after, simply select something else and reselect it to generate it.

The six visualizations offered show you the same data in different ways:

1.   'Resorts in buffer zone' - will show you your results plotted as red dots on a terrain basemap
2.   'Surrounding region' - will show the region surrounding your results with a street basemap
3. 'Avg temp by resort' - will show a bar chart of the average temperatures at each resort in your search results (in degrees Fahrenheit)
4. 'Elevation by resort' - will show a bar chart of the elevation at each resort in your search results (in US Feet)
5. 'Interactive map (suggested)' - will show a zoomable, panable webmap containing markers for each resort in your search results; click on the markers to view each resort's name and a phone number for local snow conditions
6. 'Table' - will show all data from the above visualizations in one printed table

Execute a new search at any time by changing the values in the original search boxes and clicking "Go!"

Below are some suggested search parameters for getting started:

*   Aspen, Colorado ; 80000
*   Jackson Hole, Wyoming ; 80000
*   Deer Valley, Utah ; 80000
*   Bretton Woods, New Hampshire ; 50000

In [ ]:
# data

na_skiresorts = gpd.GeoDataFrame(na_skiresorts,geometry=gpd.points_from_xy(na_skiresorts.longitude, na_skiresorts.latitude),crs=4326) # pd df to gpd gdf
na_skiresorts = na_skiresorts.set_geometry("geometry",crs="EPSG:4326") # set geo
na_skiresorts = na_skiresorts.set_crs(epsg=4326) # set crs
na_skiresorts.crs = "EPSG:4326" # direct crs
na_skiresorts = gpd.GeoDataFrame(na_skiresorts,geometry=na_skiresorts.geometry,crs=4326) # confirm

# geocoder

geocoder = Nominatim(user_agent='ski_search')

# main widgets

place_box = widgets.Text(value=input("Address: "),placeholder="Address: ",description="Address: ",disabled=False)
buffer_box = widgets.FloatText(value=float(input("Radius in meters (8,000 - 130,000): ")),description="Radius <m>: ",disabled=False)
go_button = widgets.ToggleButton(value=False,description="Go!",disabled=False,button_style="",tooltip="Description",icon="")

# top grid

grid = widgets.GridspecLayout(2,4,height="60px")
grid[0,0] = place_box
grid[0,1] = buffer_box
grid[0,2] = go_button

# explorer

def on_change(event=None):
  if buffer_box.value < 8000:
    clear_output()
    print("Please enter a radius of more than 8 kilometers (8000m).")
    display(grid)
  elif buffer_box.value > 130000:
    clear_output()
    print("Please enter a radius of less than 130 kilometers (130,000m).")
    display(grid)
  else: # proceed with valid buffer radius from input
    clear_output()
    print("Selected place: ",place_box.value)
    print("Defined radius: ",str(int(buffer_box.value))," meters")

    user_place = geocoder.geocode(place_box.value)
    user_pt = {"address":[user_place.address], "geometry":[Point(user_place.longitude, user_place.latitude)]}
    user_gdf = gpd.GeoDataFrame(user_pt, crs=4326) # user's place has been geocoded and saved to epsg:4326 'user_gdf'

    user_gdf_3310 = user_gdf.to_crs(3310) # epsg:4326 'user_gdf' reprojected and saved as epsg:3310 'user_gdf_3310' (meters as linear unit for buffering)
    buffer = user_gdf_3310.buffer(buffer_box.value) # take user's buffer distance / radius
    buffer_df = gpd.GeoDataFrame(buffer,geometry=buffer)
    buffer_df_dissolved = buffer_df.dissolve() # to gdf then dissolve

    buffer_4326 = buffer_df_dissolved.to_crs(crs="4326") # dissolved buffer back to epsg:4326 gdf 'buffer_4326' for sjoin
    buffer_4326 = buffer_4326.set_geometry("geometry",crs="EPSG:4326") # set geo
    buffer_4326 = buffer_4326.set_crs(epsg=4326) # set crs
    buffer_4326.crs = "EPSG:4326" # direct crs
    buffer_4326 = gpd.GeoDataFrame(buffer_4326, geometry=buffer_4326.geometry, crs=4326) # confirm

    buffer_4326_join = gpd.sjoin(left_df=na_skiresorts,right_df=buffer_4326,how="inner",predicate="intersects") # sjoin buffer to na_skiresorts, still epsg:4326
    buffer_4326_join = buffer_4326_join.set_geometry("geometry",crs="EPSG:4326") # set geo
    buffer_4326_join = buffer_4326_join.set_crs(epsg=4326) # set crs
    buffer_4326_join.crs = "EPSG:4326" # direct crs
    buffer_4326_join = gpd.GeoDataFrame(buffer_4326_join, geometry=buffer_4326_join.geometry, crs=4326) # confirm

    buffer_3857_temp = buffer_4326_join.to_crs(crs="3857") # from epsg:4326 project to epsg:3857 for alignment with temperature raster
    buffer_3857_temp = buffer_3857_temp.set_crs(epsg=3857) # set crs
    buffer_3857_temp.crs = "EPSG:3857" # direct crs
    buffer_3857_temp = gpd.GeoDataFrame(buffer_3857_temp, geometry=buffer_3857_temp.geometry, crs=3857) # confirm

    if len(buffer_4326_join.index) == 0: # if user's search returns no results, script will end in error; this 'if' clause allows user to retry without restarting
      print("There are no ski resorts within " + str(int(buffer_box.value)) + " meters of " + str(place_box.value) + ".")
      print("Please enter a different place or a larger radius.")
      display(grid)

    else: # query successful, visualization pending user selection
      display(grid)
      print("There are " + str(len(buffer_4326_join.index)) + " ski resorts within " + str(int(buffer_box.value)) + " meters of " + str(place_box.value) + ".") # count map
      print("Please select a visualization output. Your visualization will generate after you select something new. If you want to view the interactive map, select something else then reselect the interactive map.")
      print("Start a new search at any time.")
      vis_dropdown = widgets.Dropdown(options=['Resorts in buffer zone','Surrounding region','Avg temp by resort','Elevation by resort','Interactive map (suggested)','Table'],value='Interactive map (suggested)',description='Visualization:',disabled=False)
      display(vis_dropdown)

      ########################################################################################################################
      # make data for any of the offered visualizations
      # data originally created for maps plotted sequentially; now all data is saved here so maps can be rendered in any order

      resorts_list = [(x, y) for x, y in zip(buffer_3857_temp['geometry'].x, buffer_3857_temp['geometry'].y)] # sampling temperatures
      buffer_3857_temp['avg_temp'] = [x[0] for x in temperature.sample(resorts_list)]
      buffer_3857_temp['avg_temp_f'] = (buffer_3857_temp['avg_temp'] * (9 / 5)) + 32

      buffer_4326_dtm = buffer_3857_temp.to_crs(crs="4326") # keep 'buffer_3857_temp' gdf for new temperature columns, but project back to epsg:4326 for alignment with elevation raster
      buffer_4326_dtm = buffer_4326_dtm.set_geometry("geometry",crs="EPSG:4326") # set geo
      buffer_4326_dtm = buffer_4326_dtm.set_crs(epsg=4326) # set crs
      buffer_4326_dtm.crs = "EPSG:4326" # direct crs
      buffer_4326_dtm = gpd.GeoDataFrame(buffer_4326_dtm, geometry=buffer_4326_dtm.geometry, crs=4326) # confirm

      resorts_list = [(x, y) for x, y in zip(buffer_4326_dtm['geometry'].x, buffer_4326_dtm['geometry'].y)] # sampling elevation # redundant
      buffer_4326_dtm['elevation'] = [x[0] for x in elevation.sample(resorts_list)]
      buffer_4326_dtm['elevation_ft'] = buffer_4326_dtm['elevation'] * 3.28084

      results = buffer_4326_dtm[['name','address','avg_temp_f','elevation_ft','phone','latitude','longitude']]
      results = gpd.GeoDataFrame(results,geometry=gpd.points_from_xy(results.longitude, results.latitude),crs=4326) # create clean gdf with results
      results = results.set_geometry("geometry",crs="EPSG:4326") # set geo
      results = results.set_crs(epsg=4326) # set crs
      results.crs = "EPSG:4326" # direct crs
      results = gpd.GeoDataFrame(results,geometry=results.geometry,crs=4326) # confirm
      results = results.reset_index() # reset index

      ########################################################################################################################
      # visualization clears and redraws upon user input change

      def new_vis(event=None):
        if vis_dropdown.value == 'Resorts in buffer zone':
          # static map 1
          # plotting results
          #
          clear_output()
          display(grid)
          print("There are " + str(len(buffer_4326_join.index)) + " ski resorts within " + str(int(buffer_box.value)) + " meters of " + str(place_box.value) + ".") # count map
          print("Please select a visualization output.")
          print("Start a new search at any time.")
          display(vis_dropdown)

          fig, ax1 = plt.subplots(figsize=(12,12))
          ax1.set_title(str(len(buffer_4326_join.index)) + " ski resorts within " + str(int(buffer_box.value)) + " meters of " + str(place_box.value),fontsize=16)
          ax1.set_axis_off()
          resort_map = buffer_4326_join.plot(ax=ax1,color="Red",figsize=(12,12),markersize=36.0) # plot
          cx.add_basemap(resort_map,crs=buffer_4326_join.crs,source=cx.providers.OpenTopoMap)

        elif vis_dropdown.value == 'Surrounding region':
          # static map 2
          # road map
          #
          clear_output()
          display(grid)
          print("There are " + str(len(buffer_4326_join.index)) + " ski resorts within " + str(int(buffer_box.value)) + " meters of " + str(place_box.value) + ".") # count map
          print("Please select a visualization output.")
          print("Start a new search at any time.")
          display(vis_dropdown)

          fig, ax2 = plt.subplots(figsize=(12,12))
          ax2.set_title("SURROUNDING REGION",fontsize=16)
          ax2.set_axis_off()
          road_map = buffer_4326_join.plot(ax=ax2,color="Red",figsize=(12,12),markersize=1.0,alpha=0) # plot
          cx.add_basemap(road_map,crs=buffer_4326_join.crs,source=cx.providers.OpenStreetMap.Mapnik)

        elif vis_dropdown.value == 'Avg temp by resort':
          # static vis 1
          # resorts and temperatures
          #
          clear_output()
          display(grid)
          print("There are " + str(len(buffer_4326_join.index)) + " ski resorts within " + str(int(buffer_box.value)) + " meters of " + str(place_box.value) + ".") # count map
          print("Please select a visualization output.")
          print("Start a new search at any time.")
          display(vis_dropdown)

          name = buffer_3857_temp['name']
          avg_temp_f = buffer_3857_temp['avg_temp_f']

          fig = plt.figure(figsize = (16,8)) # avg temp over name vis
          temp_bar = plt.bar(name,avg_temp_f)
          plt.title('Average Temperature by Resort (Fahrenheit)')
          plt.show()

        elif vis_dropdown.value == 'Elevation by resort':
          # static vis 2
          # resorts and elevations
          #
          clear_output()
          display(grid)
          print("There are " + str(len(buffer_4326_join.index)) + " ski resorts within " + str(int(buffer_box.value)) + " meters of " + str(place_box.value) + ".") # count map
          print("Please select a visualization output.")
          print("Start a new search at any time.")
          display(vis_dropdown)

          name = buffer_4326_dtm['name']
          elevation_ft = buffer_4326_dtm['elevation_ft']

          fig = plt.figure(figsize = (16,8)) # elevation over name vis
          plt.bar(name,elevation_ft)
          plt.title('Elevation by Resort (US Feet)')
          plt.show()

        elif vis_dropdown.value == 'Interactive map (suggested)':
          # interactive map
          # folium
          #
          clear_output()
          display(grid)
          print("There are " + str(len(buffer_4326_join.index)) + " ski resorts within " + str(int(buffer_box.value)) + " meters of " + str(place_box.value) + ".") # count map
          print("Please select a visualization output.")
          print("Start a new search at any time.")
          display(vis_dropdown)

          fig = folium.Figure(width=1200, height=800)
          i_map = folium.Map(location=[user_place.latitude, user_place.longitude], zoom_start=9, tiles="OpenStreetMap").add_to(fig)
          fg = folium.FeatureGroup(name="results as feature")

          for index, row in results.iterrows():
            fg.add_child(folium.Marker(location=[row.latitude, row.longitude],
                                       popup=folium.Popup("This is " + str(row['name']) + ". For local snow conditions, call " + str(row.phone),parse_html=True,max_width="100%"),
                                       tooltip=folium.Tooltip(str(row['name']))
                                       ))
          i_map.add_child(fg)
          i_map
          display(fig)

        elif vis_dropdown.value == 'Table':
          clear_output()
          display(grid)
          print("There are " + str(len(buffer_4326_join.index)) + " ski resorts within " + str(int(buffer_box.value)) + " meters of " + str(place_box.value) + ".") # count map
          print("Please select a visualization output.")
          print("Start a new search at any time.")
          display(vis_dropdown)
          print(results)

      vis_dropdown.observe(new_vis,names="value")

# observe
go_button.observe(on_change,names="value")

# start
display(grid)